# CNN Replica Notebook — NFC Dataset

Adapted from the Kefra version to run on the NFC relay attack dataset.

Goal: same CNN approach and evaluation structure, applied to NFC ATQA amplitude data.

Key similarities to original notebook:
- Three-class supervised classification: normal, wired_relay, wireless_relay
- 90/10 train/test split
- 5-fold cross-validation
- Optimizer comparison: SGD, Adam, RMSProp
- Learning rate 0.001
- 100 epochs
- Mini-batch size 8

Key differences from original notebook:
- Input: 1800 amplitude samples per row (was 512)
- Classes: normal / wired_relay / wireless_relay (was Real / Fake High / Fake Low)
- Data loaded from nfc_atqa.csv (or split CSVs) instead of Kefra CSVs
- No augmentation step (66,366 samples already sufficient)


In [6]:
# ==============================
# Block 1: Data loading
# ==============================

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

# ------------------------------
# Reproducibility
# ------------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ------------------------------
# Configure path
# ------------------------------

# Option A: single combined CSV  ← use this if you ran wav_to_tabular.py without --split
CSV_PATH = Path("data/nfc_atqa.csv")

# Option B: split CSVs  ← use this if you ran with --split
# CSV_PATH = None   # comment out the line above and uncomment this
# SPLIT_DIR = Path("data")

FEATURE_COLS = [f"x{i}" for i in range(1800)]
N_FEATURES   = 1800

# ------------------------------
# Load
# ------------------------------

if CSV_PATH is not None and CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH)
else:
    split_files = sorted(Path("data").glob("nfc_atqa_*.csv"))
    df = pd.concat([pd.read_csv(f) for f in split_files], ignore_index=True)
    print(f"Loaded {len(split_files)} split files")

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df["class"].value_counts())

# ------------------------------
# Build X, y
# ------------------------------

X = df[FEATURE_COLS].to_numpy(dtype=np.float32)

# Encode classes to integers: normal=0, wired_relay=1, wireless_relay=2
class_names = ["normal", "wired_relay", "wireless_relay"]
class_to_idx = {c: i for i, c in enumerate(class_names)}
y = df["class"].map(class_to_idx).to_numpy(dtype=np.int64)

files = df["filepath"].copy() if "filepath" in df.columns else pd.Series([f"sample_{i}" for i in range(len(df))])

# Fully shuffle before the 90/10 split
rng  = np.random.default_rng(SEED)
perm = rng.permutation(len(X))
X     = X[perm]
y     = y[perm]
files = files.iloc[perm].reset_index(drop=True)

assert X.shape[1] == N_FEATURES, f"Expected {N_FEATURES} columns, got {X.shape[1]}"

print(f"\nCombined X: {X.shape}")
print(f"Combined class counts: {pd.Series(y).value_counts().sort_index().to_dict()}")


Dataset shape: (66366, 1808)

Class distribution:
class
normal            29496
wired_relay       29496
wireless_relay     7374
Name: count, dtype: int64

Combined X: (66366, 1800)
Combined class counts: {0: 29496, 1: 29496, 2: 7374}


In [7]:
# ==============================
# Block 2: 1D CNN model, training, testing functions
# ==============================

import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

BATCH_SIZE    = 1024    # RTX 5070 
LEARNING_RATE = 1e-3    # same as paper
NUM_EPOCHS    = 100     # same as paper


def make_activation(name: str):
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    if name == "leakyrelu":
        return nn.LeakyReLU(negative_slope=0.01)
    if name == "elu":
        return nn.ELU()
    raise ValueError(f"Unknown activation: {name}")


class CNN1DClassifier(nn.Module):
    """Small 1D CNN classifier for 1800-point NFC amplitude vectors."""

    def __init__(self, num_classes=3, activation="relu"):
        super().__init__()
        act1 = make_activation(activation)
        act2 = make_activation(activation)
        act3 = make_activation(activation)
        act4 = make_activation(activation)

        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3),   # 1800 -> 900
            act1,
            nn.BatchNorm1d(16),

            nn.Conv1d(16, 32, kernel_size=5, stride=2, padding=2),  # 900 -> 450
            act2,
            nn.BatchNorm1d(32),

            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),  # 450 -> 225
            act3,
            nn.BatchNorm1d(64),

            nn.Conv1d(64, 128, kernel_size=3, stride=2, padding=1), # 225 -> 113
            act4,
            nn.BatchNorm1d(128),

            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            make_activation(activation),
            nn.Dropout(0.20),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def make_optimizer(name: str, model: nn.Module, lr: float):
    name = name.lower()
    if name == "sgd":
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    if name == "adam":
        return optim.Adam(model.parameters(), lr=lr)
    if name == "rmsprop":
        return optim.RMSprop(model.parameters(), lr=lr, momentum=0.9)
    raise ValueError(f"Unknown optimizer: {name}")


def scale_and_tensorize(X_train, X_test):
    # Fit scaler on training only to avoid leakage.
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train).astype(np.float32)
    X_test_s  = scaler.transform(X_test).astype(np.float32)

    # Conv1d expects [batch, channels, length].
    X_train_t = torch.tensor(X_train_s).unsqueeze(1)
    X_test_t  = torch.tensor(X_test_s).unsqueeze(1)
    return X_train_t, X_test_t, scaler


def make_loader(X_tensor, y_array, batch_size=BATCH_SIZE, shuffle=False):
    y_tensor = torch.tensor(y_array, dtype=torch.long)
    return DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=shuffle)


def train_model(X_train, y_train, X_test, y_test, optimizer_name="sgd", activation="relu", verbose=False):
    X_train_t, X_test_t, scaler = scale_and_tensorize(X_train, X_test)
    train_loader = make_loader(X_train_t, y_train, shuffle=True)
    test_loader  = make_loader(X_test_t,  y_test,  shuffle=False)

    model     = CNN1DClassifier(num_classes=3, activation=activation).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(optimizer_name, model, LEARNING_RATE)

    train_losses = []

    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss   = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_loss)

        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            print(f"Epoch {epoch+1:03d}/{NUM_EPOCHS} | loss={epoch_loss:.6f}")

    y_pred, y_prob = predict_model(model, test_loader)

    return {
        "model":        model,
        "scaler":       scaler,
        "train_losses": train_losses,
        "y_pred":       y_pred,
        "y_prob":       y_prob,
    }


def predict_model(model, loader):
    model.eval()
    preds = []
    probs = []

    with torch.no_grad():
        for xb, _ in loader:
            xb     = xb.to(device)
            logits = model(xb)
            p      = torch.softmax(logits, dim=1)
            pred   = torch.argmax(p, dim=1)
            preds.extend(pred.cpu().numpy())
            probs.extend(p.cpu().numpy())

    return np.array(preds), np.array(probs)


def summarize_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "accuracy":        acc,
        "macro_precision": precision,
        "macro_recall":    recall,
        "macro_f1":        f1,
    }


Using device: cuda


In [8]:
# ==============================
# Block 3: 90/10 test split experiments
# ==============================

X_train, X_test, y_train, y_test, files_train, files_test = train_test_split(
    X, y, files,
    test_size=0.10,
    random_state=SEED,
    shuffle=True,
    stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train counts:", pd.Series(y_train).value_counts().sort_index().to_dict())
print("Test counts:",  pd.Series(y_test).value_counts().sort_index().to_dict())

optimizers_to_test  = ["sgd", "adam", "rmsprop"]
activations_to_test = ["relu", "leakyrelu", "elu"]

results_90_10 = []
trained_runs  = {}

for opt_name in optimizers_to_test:
    for act_name in activations_to_test:
        print(f"\nTraining 90/10 model: optimizer={opt_name}, activation={act_name}")
        run = train_model(
            X_train, y_train, X_test, y_test,
            optimizer_name=opt_name,
            activation=act_name,
            verbose=False
        )
        y_pred  = run["y_pred"]
        metrics = summarize_metrics(y_test, y_pred)
        metrics.update({"optimizer": opt_name, "activation": act_name})
        results_90_10.append(metrics)
        trained_runs[(opt_name, act_name)] = run
        print(metrics)

results_90_10_df = pd.DataFrame(results_90_10).sort_values(
    by=["accuracy", "macro_f1"], ascending=False
).reset_index(drop=True)

results_90_10_df


Train shape: (59729, 1800) Test shape: (6637, 1800)
Train counts: {0: 26546, 1: 26546, 2: 6637}
Test counts: {0: 2950, 1: 2950, 2: 737}

Training 90/10 model: optimizer=sgd, activation=relu
{'accuracy': 1.0, 'macro_precision': 1.0, 'macro_recall': 1.0, 'macro_f1': 1.0, 'optimizer': 'sgd', 'activation': 'relu'}

Training 90/10 model: optimizer=sgd, activation=leakyrelu
{'accuracy': 0.9996986590326955, 'macro_precision': 0.9997741644083108, 'macro_recall': 0.9990954319312527, 'macro_f1': 0.9994341453891827, 'optimizer': 'sgd', 'activation': 'leakyrelu'}

Training 90/10 model: optimizer=sgd, activation=elu
{'accuracy': 0.9870423384059063, 'macro_precision': 0.9697546118199026, 'macro_recall': 0.9848538509302487, 'macro_f1': 0.9769350472624397, 'optimizer': 'sgd', 'activation': 'elu'}

Training 90/10 model: optimizer=adam, activation=relu
{'accuracy': 1.0, 'macro_precision': 1.0, 'macro_recall': 1.0, 'macro_f1': 1.0, 'optimizer': 'adam', 'activation': 'relu'}

Training 90/10 model: optimiz

,accuracy,macro_precision,macro_recall,macro_f1,optimizer,activation
0,1.000000,1.000000,1.000000,1.000000,sgd,relu
1,1.000000,1.000000,1.000000,1.000000,adam,relu
2,1.000000,1.000000,1.000000,1.000000,adam,leakyrelu
3,1.000000,1.000000,1.000000,1.000000,adam,elu
4,1.000000,1.000000,1.000000,1.000000,rmsprop,relu
5,1.000000,1.000000,1.000000,1.000000,rmsprop,leakyrelu
6,1.000000,1.000000,1.000000,1.000000,rmsprop,elu
7,0.999699,0.999774,0.999095,0.999434,sgd,leakyrelu
8,0.987042,0.969755,0.984854,0.976935,sgd,elu


In [9]:
# Detailed report for the best 90/10 model

best     = results_90_10_df.iloc[0]
best_key = (best["optimizer"], best["activation"])
best_run = trained_runs[best_key]

print("Best 90/10 configuration:")
print(best)

print("\nClassification report:")
print(classification_report(y_test, best_run["y_pred"], target_names=class_names, zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_test, best_run["y_pred"]))


Best 90/10 configuration:
accuracy            1.0
macro_precision     1.0
macro_recall        1.0
macro_f1            1.0
optimizer           sgd
activation         relu
Name: 0, dtype: object

Classification report:
                precision    recall  f1-score   support

        normal       1.00      1.00      1.00      2950
   wired_relay       1.00      1.00      1.00      2950
wireless_relay       1.00      1.00      1.00       737

      accuracy                           1.00      6637
     macro avg       1.00      1.00      1.00      6637
  weighted avg       1.00      1.00      1.00      6637

Confusion matrix:
[[2950    0    0]
 [   0 2950    0]
 [   0    0  737]]


In [10]:
# ==============================
# Block 4: 5-fold cross-validation
# ==============================

# Each fold refits the scaler only on the training fold to avoid leakage.

skf        = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = []

for opt_name in optimizers_to_test:
    for act_name in activations_to_test:
        print(f"\n5-fold CV: optimizer={opt_name}, activation={act_name}")

        for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]

            run = train_model(
                X_tr, y_tr, X_te, y_te,
                optimizer_name=opt_name,
                activation=act_name,
                verbose=False
            )

            metrics = summarize_metrics(y_te, run["y_pred"])
            metrics.update({
                "optimizer":  opt_name,
                "activation": act_name,
                "fold":       fold,
            })
            cv_results.append(metrics)
            print(f"  fold={fold} acc={metrics['accuracy']:.4f} f1={metrics['macro_f1']:.4f}")

cv_results_df = pd.DataFrame(cv_results)

cv_summary_df = (
    cv_results_df
    .groupby(["optimizer", "activation"])
    .agg(
        accuracy_mean=("accuracy",         "mean"),
        accuracy_std= ("accuracy",         "std"),
        macro_precision_mean=("macro_precision", "mean"),
        macro_recall_mean=   ("macro_recall",    "mean"),
        macro_f1_mean=       ("macro_f1",        "mean"),
        macro_f1_std=        ("macro_f1",        "std"),
    )
    .reset_index()
    .sort_values(by=["accuracy_mean", "macro_f1_mean"], ascending=False)
)

cv_summary_df



5-fold CV: optimizer=sgd, activation=relu
  fold=1 acc=0.9844 f1=0.9693
  fold=2 acc=0.9999 f1=0.9999
  fold=3 acc=0.9998 f1=0.9997
  fold=4 acc=0.9989 f1=0.9979
  fold=5 acc=0.9998 f1=0.9996

5-fold CV: optimizer=sgd, activation=leakyrelu
  fold=1 acc=0.9997 f1=0.9994
  fold=2 acc=0.9928 f1=0.9871
  fold=3 acc=0.9953 f1=0.9915
  fold=4 acc=0.9990 f1=0.9982
  fold=5 acc=0.9992 f1=0.9986

5-fold CV: optimizer=sgd, activation=elu
  fold=1 acc=0.8954 f1=0.6948
  fold=2 acc=0.9661 f1=0.9310
  fold=3 acc=0.9351 f1=0.8928
  fold=4 acc=0.9624 f1=0.9277
  fold=5 acc=0.9604 f1=0.9200

5-fold CV: optimizer=adam, activation=relu
  fold=1 acc=1.0000 f1=1.0000
  fold=2 acc=1.0000 f1=1.0000
  fold=3 acc=0.9999 f1=0.9999
  fold=4 acc=1.0000 f1=1.0000
  fold=5 acc=1.0000 f1=1.0000

5-fold CV: optimizer=adam, activation=leakyrelu
  fold=1 acc=0.9999 f1=0.9999
  fold=2 acc=1.0000 f1=1.0000
  fold=3 acc=0.9998 f1=0.9998
  fold=4 acc=1.0000 f1=1.0000
  fold=5 acc=1.0000 f1=1.0000

5-fold CV: optimizer=ad

,optimizer,activation,accuracy_mean,accuracy_std,macro_precision_mean,macro_recall_mean,macro_f1_mean,macro_f1_std
2,adam,relu,0.999985,0.000034,0.999989,0.999955,0.999972,0.000063
3,rmsprop,elu,0.999985,0.000034,0.999989,0.999955,0.999972,0.000063
1,adam,leakyrelu,0.999955,0.000067,0.999966,0.999898,0.999932,0.000095
0,adam,elu,0.999864,0.000112,0.999831,0.999661,0.999746,0.000210
4,rmsprop,leakyrelu,0.999397,0.001306,0.998398,0.999514,0.998946,0.002278
7,sgd,leakyrelu,0.997227,0.003010,0.993111,0.997006,0.994967,0.005398
8,sgd,relu,0.996565,0.006811,0.997222,0.990034,0.993270,0.013413
5,rmsprop,relu,0.995660,0.009704,0.989143,0.996711,0.992412,0.016967
6,sgd,elu,0.943873,0.029747,0.914513,0.871184,0.873256,0.100871


In [11]:
# Save results

results_90_10_df.to_csv("nfc_cnn_90_10_results.csv",    index=False)
cv_results_df.to_csv(   "nfc_cnn_5fold_raw_results.csv", index=False)
cv_summary_df.to_csv(   "nfc_cnn_5fold_summary.csv",     index=False)

print("Saved:")
print("- nfc_cnn_90_10_results.csv")
print("- nfc_cnn_5fold_raw_results.csv")
print("- nfc_cnn_5fold_summary.csv")


Saved:
- nfc_cnn_90_10_results.csv
- nfc_cnn_5fold_raw_results.csv
- nfc_cnn_5fold_summary.csv
